# `submission_05` — финальная модель (лучший приватный результат)

**Итог:** Private macro-F1 = **0.84958** · Public = 0.84978 — лучший из всех сабмитов по приватному лидерборду.

**Суть подхода (два рычага):**
1. **Модельная импутация пропущенного EOG.** Признак `eog_burst_index` отсутствует у ~50% эпох. Вместо удаления/медианы он восстанавливается регрессией по остальным признакам (`IterativeImputer` + `BayesianRidge`). Это главный прирост: CV 0.836 → 0.844.
2. **Разнообразный ансамбль (soft-voting).** SVC + HistGradientBoosting + ExtraTrees + 2×MLP, усреднение по 3 сидам (seed-bagging).

Всё валидировалось 5-fold stratified CV (мультисид), импьютер и модели обучаются внутри пайплайна (без утечки). Ноутбук воспроизводит `submission_05.csv`.

In [ ]:
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.ensemble import HistGradientBoostingClassifier, ExtraTreesClassifier, VotingClassifier

names={0:"Wake",1:"Light",2:"Deep",3:"REM"}
tr=pd.read_csv("train.csv"); te=pd.read_csv("test.csv"); ss=pd.read_csv("sample_submission.csv")
feat=[c for c in tr.columns if c not in ("id","sleep_stage")]; y=tr.sleep_stage.values
print("train",tr.shape,"test",te.shape,"| EOG пропущен: train %.0f%%, test %.0f%%"%(
    tr["eog_burst_index"].isna().mean()*100, te["eog_burst_index"].isna().mean()*100))

**Инженерия признаков:** относительные мощности ЭЭГ-диапазонов, отношения delta/beta и theta/alpha, сумма медленных волн и бинарный флаг пропуска EOG.

In [ ]:
EEG=["eeg_delta_power","eeg_theta_power","eeg_alpha_power","eeg_sigma_power","eeg_beta_power","eeg_gamma_power"]
def fe(df):
    X=df.copy(); tot=X[EEG].clip(lower=0).sum(1)+1e-6
    for b in EEG: X["rel_"+b]=X[b]/tot
    X["delta_beta"]=X["eeg_delta_power"]/(X["eeg_beta_power"].abs()+1e-6)
    X["theta_alpha"]=X["eeg_theta_power"]/(X["eeg_alpha_power"].abs()+1e-6)
    X["slow_dom"]=X["eeg_slow_osc_power"]+X["eeg_delta_power"]
    X["eog_burst_missing"]=df["eog_burst_index"].isna().astype(int)   # флаг: значение было пропущено
    return X
X=fe(tr[feat]); X_test=fe(te[feat])
print("признаков после FE:", X.shape[1])

**Модель.** `IterativeImputer` восстанавливает EOG, затем soft-voting ансамбль из 5 моделей.
SVC и MLP получают стандартизированные признаки (StandardScaler); деревьям масштаб не нужен.

In [ ]:
def II(): return IterativeImputer(estimator=BayesianRidge(), max_iter=5, random_state=42)
def sc(m): return Pipeline([("s",StandardScaler()),("m",m)])

def v5(seed):
    return Pipeline([("imp", II()), ("vote", VotingClassifier([
        ("svc",  sc(SVC(C=80, gamma=0.008, probability=True, random_state=seed))),
        ("hgb",  HistGradientBoostingClassifier(random_state=seed, learning_rate=0.079,
                    max_iter=240, max_leaf_nodes=43, min_samples_leaf=24, l2_regularization=7.26)),
        ("et",   ExtraTreesClassifier(n_estimators=430, max_features=0.89,
                    min_samples_leaf=1, random_state=seed, n_jobs=-1)),
        ("mlp1", sc(MLPClassifier(hidden_layer_sizes=(128,64), alpha=1e-3,
                    max_iter=400, early_stopping=True, random_state=seed))),
        ("mlp2", sc(MLPClassifier(hidden_layer_sizes=(200,100), activation="tanh", alpha=1e-3,
                    max_iter=400, early_stopping=True, random_state=seed)))],
        voting="soft", n_jobs=-1))])

**Проверка качества (кросс-валидация).** 5-fold stratified, метрика macro-F1. Ожидаемо ≈ 0.844.

In [ ]:
cv = cross_val_score(v5(42), X, y, cv=StratifiedKFold(5, shuffle=True, random_state=42),
                     scoring="f1_macro", n_jobs=-1).mean()
print(f"CV macro-F1 (seed 42) = {cv:.4f}")

**Сборка сабмита.** Seed-bagging: обучаем на полном train для сидов [0, 1, 42], усредняем вероятности, берём argmax.

In [ ]:
SEEDS=[0,1,42]
probs=np.zeros((len(X_test),4))
for s in SEEDS:
    m=v5(s); m.fit(X,y); probs+=m.predict_proba(X_test)
pred=(probs/len(SEEDS)).argmax(1)

sub=pd.DataFrame({"id":te.id, "sleep_stage":pred})
sub.to_csv("submission_05.csv", index=False)
ok=(list(sub.columns)==["id","sleep_stage"] and len(sub)==len(ss)
    and (sub.id.values==ss.id.values).all() and set(sub.sleep_stage)<=set([0,1,2,3]))
print("submission_05.csv записан. формат OK =", bool(ok))
print("распределение:", [f'{names[i]}={(pred==i).mean()*100:.1f}%' for i in range(4)])

## Выводы
- Главный приём — **восстановление пропущенного EOG** регрессией, а не его удаление.
- **Ансамбль** разных моделей + усреднение по сидам дают устойчивый результат.
- Отбор моделей и приёмов вёлся **по кросс-валидации**, а не по публичному борду — поэтому модель оказалась лучшей на **приватном** лидерборде (0.84958).